<a href="https://colab.research.google.com/github/PMoshel/data-on-conflicts-and-spatial-characteristics-of-the-territory/blob/main/algorithm2_comparison_of_territories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Шаг 1: загрузка и проверка данных о территориях

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Настройки отображения pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.max_colwidth', 50)

# Загрузка данных
from google.colab import files
import os

# Проверяем наличие файла territory.xlsx, если нет – просим загрузить
if not os.path.exists('territory.xlsx'):
    print("Файл territory.xlsx не найден.")
    print("Пожалуйста, загрузите базу данных пространственно-функциональных характеристик (файл Excel с именем territory.xlsx).")
    uploaded = files.upload()
    # Если загруженный файл имеет другое имя, переименовываем его (на всякий случай)
    for fn in uploaded.keys():
        if fn != 'territory.xlsx':
            os.rename(fn, 'territory.xlsx')
            print(f"Файл {fn} переименован в territory.xlsx")

# Загрузка данных
df = pd.read_excel('territory.xlsx')

# Порядок и названия столбцов
new_column_order = [
    'ID',
    'Триггер',
    'Существующее функциональное использование территории',
    'Планируемое функциональное использование территории',
    'Функциональное использование окружения',
    'Год застройки окружения',
    'Коэффициент застройки',
    'Коэффициент центральности',
    'Население',
    'Год постройки ОКС',
    'Новое строительство',
    'Снос',
    'Исторические здания',
    'Железная дорога',
    'Автодорога',
    'Инженерные объекты',
    'Водные объекты'
]

# Оставляем только нужные столбцы и задаём порядок
df = df[new_column_order]

# Функция для очистки пробелов в данных
def clean_data(df):
    """ Очищает данные от лишних пробелов в текстовых столбцах и приводит к нижнему регистру """
    df_clean = df.copy()

    # Список текстовых столбцов для очистки
    text_columns = [
        'Существующее функциональное использование территории',
        'Планируемое функциональное использование территории',
        'Функциональное использование окружения'
    ]

    for col in text_columns:
        if col in df_clean.columns:
            # Приводим к нижнему регистру
            df_clean[col] = df_clean[col].astype(str).str.lower()
            # Удаляем пробелы в начале и конце строк
            df_clean[col] = df_clean[col].str.strip()
            # Заменяем множественные пробелы на один
            df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)

    # Числовые столбцы (включая бинарные как числа)
    numeric_columns = [
        'Год застройки окружения',
        'Коэффициент застройки',
        'Коэффициент центральности',
        'Население',
        'Год постройки ОКС',
        'Новое строительство',
        'Снос',
        'Исторические здания',
        'Железная дорога',
        'Автодорога',
        'Инженерные объекты',
        'Водные объекты'
    ]

    for col in numeric_columns:
        if col in df_clean.columns:
            # Если значения строковые (с пробелами), преобразуем
            if df_clean[col].dtype == 'object':
                # Удаляем пробелы и преобразуем в числа
                df_clean[col] = pd.to_numeric(
                    df_clean[col].astype(str).str.replace(' ', ''),
                    errors='coerce'
                )

     # Преобразуем целочисленные столбцы с сохранением <NA>
    integer_columns = ['Год застройки окружения','Население', 'Год постройки ОКС', 'Новое строительство', 'Снос', 'Исторические здания', 'Железная дорога', 'Автодорога', 'Инженерные объекты', 'Водные объекты']
    for col in integer_columns:
        if col in df_clean.columns:
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                df_clean[col] = df_clean[col].astype('Int64')

    return df_clean

# Очищаем данные
df = clean_data(df)

# 1. Первые 5 строк данных
print("Первые 5 строк данных:")
print(df.head())
print("\n" + "="*80)

# 2. Компактная информация о данных
print("\nОБЩАЯ ИНФОРМАЦИЯ О ДАННЫХ:")
print(f"Количество строк: {df.shape[0]}")
print(f"Количество столбцов: {df.shape[1]}")
print("\nТИПЫ ДАННЫХ И ПРОПУСКИ:")
info_df = pd.DataFrame({
    'Тип данных': df.dtypes,
    'Не пропущено': df.count(),
    'Пропущено': df.isnull().sum(),
    '% пропусков': (df.isnull().sum() / len(df) * 100).round(1)
})
print(info_df)
print("\n" + "="*80)

# Сохраняем копию данных
data = df.copy()

# Список столбцов для анализа (все кроме ID и Триггера)
analysis_columns = [col for col in data.columns if col not in ['ID', 'Триггер']]
print("\nВведите количественно-качественные характеристики проектируемой территории.")
print(f"\nСтолбцы для анализа ({len(analysis_columns)}): {', '.join(analysis_columns)}")


Шаг 2: существующее функциональное использование территории

In [ ]:
# Код для сравнения по столбцу "Существующее функциональное использование территории"

# 1. Выводим список функций
def print_categories():
    categories = [
        "Акватория (береговые полосы, острова, пляжи, водоемы)",
        "Другая (строительство, сложно классифицировать)",
        "Другая негативная (кладбища, инженерные сооружения, свалки)",
        "Жилая (жилые дома, общежития, СНТ)",
        "Коммерческая (магазины, кафе, рестораны, рынки)",
        "Культурно-досуговая (театры, кино, библиотеки, музеи)",
        "Ландшафтно-рекреационная (парки, скверы, бульвары)",
        "Общественно-деловая (банки, офисы, административные здания)",
        "Промышленная (заводы, фабрики)",
        "Рекреационная (игровые площадки, спортивные объекты, места для отдыха)",
        "Сельскохозяйственная (поля, фермы)",
        "Социальная (школы, больницы, детские сады, университеты)",
        "Транспортная (парковки, гаражи, заправки)",
        "Экологически-природная (луга, пустыри, природные территории)",
        "Экологически-природная c активно растущими деревьями (леса, заросли)"
    ]

    for category in categories:
        print(category)

# Вызов функции
print("ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print("(можно ввести несколько функций через запятую)")
print_categories()
print("\n" + "="*80)

# 2. Пользователь вводит функцию для сравнения
print("\nВВЕДИТЕ СУЩЕСТВУЮЩЕЕ ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print("(можно ввести несколько функций через запятую)")
user_input = input(">>> ").strip()

# Разделяем введенные функции
user_functions = [f.strip().lower() for f in user_input.split(',') if f.strip()]
print(f"\nПоиск по функциям: {user_functions}")
print("\n"+"="*80)

# 3. Матрица схожести функций (без изменений)
similarity_matrix = {
    'акватория': {
        'акватория': 1.0,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.1,
        'коммерческая': 0.2,
        'культурно-досуговая': 0.3,
        'ландшафтно-рекреационная': 0.5,
        'общественно-деловая': 0.0,
        'промышленная': 0.0,
        'рекреационная': 0.4,
        'сельскохозяйственная': 0.3,
        'социальная': 0.1,
        'транспортная': 0.0,
        'экологически-природная': 0.4,
        'экологически-природная c активно растущими деревьями': 0.2
    },

    'другая': {
        'акватория': 0.1,
        'другая': 1.0,
        'другая негативная': 0.3,
        'жилая': 0.1,
        'коммерческая': 0.2,
        'культурно-досуговая': 0.1,
        'ландшафтно-рекреационная': 0.1,
        'общественно-деловая': 0.2,
        'промышленная': 0.2,
        'рекреационная': 0.1,
        'сельскохозяйственная': 0.1,
        'социальная': 0.1,
        'транспортная': 0.2,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'другая негативная': {
        'акватория': 0.0,
        'другая': 0.3,
        'другая негативная': 1.0,
        'жилая': 0.0,
        'коммерческая': 0.0,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.0,
        'общественно-деловая': 0.1,
        'промышленная': 0.5,
        'рекреационная': 0.0,
        'сельскохозяйственная': 0.0,
        'социальная': 0.0,
        'транспортная': 0.3,
        'экологически-природная': 0.0,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'жилая': {
        'акватория': 0.1,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 1.0,
        'коммерческая': 0.5,
        'культурно-досуговая': 0.3,
        'ландшафтно-рекреационная': 0.3,
        'общественно-деловая': 0.4,
        'промышленная': 0.0,
        'рекреационная': 0.3,
        'сельскохозяйственная': 0.2,
        'социальная': 0.7,
        'транспортная': 0.2,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'коммерческая': {
        'акватория': 0.2,
        'другая': 0.2,
        'другая негативная': 0.0,
        'жилая': 0.5,
        'коммерческая': 1.0,
        'культурно-досуговая': 0.7,
        'ландшафтно-рекреационная': 0.2,
        'общественно-деловая': 0.8,
        'промышленная': 0.1,
        'рекреационная': 0.4,
        'сельскохозяйственная': 0.1,
        'социальная': 0.3,
        'транспортная': 0.4,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'культурно-досуговая': {
        'акватория': 0.3,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.3,
        'коммерческая': 0.7,
        'культурно-досуговая': 1.0,
        'ландшафтно-рекреационная': 0.4,
        'общественно-деловая': 0.6,
        'промышленная': 0.0,
        'рекреационная': 0.7,
        'сельскохозяйственная': 0.0,
        'социальная': 0.5,
        'транспортная': 0.0,
        'экологически-природная': 0.2,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'ландшафтно-рекреационная': {
        'акватория': 0.5,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.3,
        'коммерческая': 0.2,
        'культурно-досуговая': 0.4,
        'ландшафтно-рекреационная': 1.0,
        'общественно-деловая': 0.2,
        'промышленная': 0.0,
        'рекреационная': 0.7,
        'сельскохозяйственная': 0.3,
        'социальная': 0.3,
        'транспортная': 0.0,
        'экологически-природная': 0.7,
        'экологически-природная c активно растущими деревьями': 0.7
    },

    'общественно-деловая': {
        'акватория': 0.0,
        'другая': 0.2,
        'другая негативная': 0.1,
        'жилая': 0.4,
        'коммерческая': 0.8,
        'культурно-досуговая': 0.6,
        'ландшафтно-рекреационная': 0.2,
        'общественно-деловая': 1.0,
        'промышленная': 0.1,
        'рекреационная': 0.3,
        'сельскохозяйственная': 0.1,
        'социальная': 0.6,
        'транспортная': 0.3,
        'экологически-природная': 0.1,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'промышленная': {
        'акватория': 0.0,
        'другая': 0.2,
        'другая негативная': 0.5,
        'жилая': 0.0,
        'коммерческая': 0.1,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.0,
        'общественно-деловая': 0.1,
        'промышленная': 1.0,
        'рекреационная': 0.0,
        'сельскохозяйственная': 0.0,
        'социальная': 0.0,
        'транспортная': 0.7,
        'экологически-природная': 0.0,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'рекреационная': {
        'акватория': 0.4,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.3,
        'коммерческая': 0.4,
        'культурно-досуговая': 0.7,
        'ландшафтно-рекреационная': 0.7,
        'общественно-деловая': 0.3,
        'промышленная': 0.0,
        'рекреационная': 1.0,
        'сельскохозяйственная': 0.1,
        'социальная': 0.5,
        'транспортная': 0.0,
        'экологически-природная': 0.5,
        'экологически-природная c активно растущими деревьями': 0.4
    },

    'сельскохозяйственная': {
        'акватория': 0.3,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.2,
        'коммерческая': 0.1,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.3,
        'общественно-деловая': 0.1,
        'промышленная': 0.0,
        'рекреационная': 0.1,
        'сельскохозяйственная': 1.0,
        'социальная': 0.1,
        'транспортная': 0.0,
        'экологически-природная': 0.5,
        'экологически-природная c активно растущими деревьями': 0.5
    },

    'социальная': {
        'акватория': 0.1,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.7,
        'коммерческая': 0.3,
        'культурно-досуговая': 0.5,
        'ландшафтно-рекреационная': 0.3,
        'общественно-деловая': 0.6,
        'промышленная': 0.0,
        'рекреационная': 0.5,
        'сельскохозяйственная': 0.1,
        'социальная': 1.0,
        'транспортная': 0.4,
        'экологически-природная': 0.2,
        'экологически-природная c активно растущими деревьями': 0.1
    },

    'транспортная': {
        'акватория': 0.0,
        'другая': 0.2,
        'другая негативная': 0.3,
        'жилая': 0.2,
        'коммерческая': 0.4,
        'культурно-досуговая': 0.0,
        'ландшафтно-рекреационная': 0.0,
        'общественно-деловая': 0.3,
        'промышленная': 0.7,
        'рекреационная': 0.0,
        'сельскохозяйственная': 0.0,
        'социальная': 0.4,
        'транспортная': 1.0,
        'экологически-природная': 0.0,
        'экологически-природная c активно растущими деревьями': 0.0
    },

    'экологически-природная': {
        'акватория': 0.4,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.1,
        'коммерческая': 0.1,
        'культурно-досуговая': 0.2,
        'ландшафтно-рекреационная': 0.7,
        'общественно-деловая': 0.1,
        'промышленная': 0.0,
        'рекреационная': 0.5,
        'сельскохозяйственная': 0.5,
        'социальная': 0.2,
        'транспортная': 0.0,
        'экологически-природная': 1.0,
        'экологически-природная c активно растущими деревьями': 0.9
    },

    'экологически-природная c активно растущими деревьями': {
        'акватория': 0.2,
        'другая': 0.1,
        'другая негативная': 0.0,
        'жилая': 0.1,
        'коммерческая': 0.0,
        'культурно-досуговая': 0.1,
        'ландшафтно-рекреационная': 0.7,
        'общественно-деловая': 0.0,
        'промышленная': 0.0,
        'рекреационная': 0.4,
        'сельскохозяйственная': 0.5,
        'социальная': 0.1,
        'транспортная': 0.0,
        'экологически-природная': 0.9,
        'экологически-природная c активно растущими деревьями': 1.0
    }
}

# 4. Функция для расчета схожести функций (исправлена)
def calculate_function_similarity(func1, func2):
    """Рассчитывает схожесть между двумя функциями"""
    # Если хотя бы одна функция отсутствует - возвращаем None
    if pd.isna(func1) or pd.isna(func2):
        return None

    # Точное совпадение
    if func1 == func2:
        return 1.0

    # Проверяем матрицу схожести
    if func1 in similarity_matrix and func2 in similarity_matrix[func1]:
        return similarity_matrix[func1][func2]
    elif func2 in similarity_matrix and func1 in similarity_matrix[func2]:
        return similarity_matrix[func2][func1]

    # Если функция не найдена в матрице — сходство нулевое
    return 0.0

# 5. Рассчитываем схожесть для каждой строки
results = []
for idx, row in data.iterrows():
    territory_func = row['Существующее функциональное использование территории']

    # Если значение территории отсутствует — пропускаем (None)
    if pd.isna(territory_func):
        similarity = None
    # Если пользователь ничего не ввел
    elif not user_functions:
        similarity = None
    else:
        # Рассчитываем максимальную схожесть с введенными функциями
        max_similarity = None
        for user_func in user_functions:
            sim = calculate_function_similarity(territory_func, user_func)
            if sim is not None:
                if max_similarity is None or sim > max_similarity:
                    max_similarity = sim

        # Если все введённые функции некорректны → 0%
        similarity = max_similarity * 100 if max_similarity is not None else 0.0

    results.append({
        'ID': row['ID'],
        'Сходство %': round(similarity, 1) if similarity is not None else None,
        'Существующее функциональное использование территории': territory_func,
        'Триггер': row['Триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 6. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 МАКСИМАЛЬНО ПОХОЖИХ ТЕРРИТОРИЙ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 7. Сохраняем результаты для финального сравнения
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Существующее функциональное использование территории'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


Шаг 3: планируемое функциональное использование территории

In [ ]:
# Код для сравнения по столбцу "Планируемое функциональное использование территории"

# 1. Выводим список функций
print("ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print_categories()
print("\n" + "="*80)

# 2. Пользователь вводит планируемые функции
print("\nВВЕДИТЕ ПЛАНИРУЕМОЕ ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print("(можно ввести несколько функций через запятую)")
user_input = input(">>> ").strip()

# Разделяем введенные функции
user_functions = [f.strip().lower() for f in user_input.split(',') if f.strip()]
print(f"\nПоиск по функциям: {user_functions}")
print("\n"+"="*80)

# 3. Проверяем, существует ли матрица сходства (из шага 2)
if 'similarity_matrix' not in globals():
    # Если по какой-то причине матрицы нет – выводим ошибку и завершаем
    raise ValueError("Матрица сходства не найдена. Сначала выполните шаг 2 (существующее использование).")

# 4. Используем ту же функцию для расчета схожести
def calculate_function_similarity(func1, func2):
    """Рассчитывает схожесть между двумя функциями"""
    # Если хотя бы одна функция отсутствует - возвращаем None
    if pd.isna(func1) or pd.isna(func2):
        return None

    # Точное совпадение
    if func1 == func2:
        return 1.0

    # Проверяем матрицу схожести
    if func1 in similarity_matrix and func2 in similarity_matrix[func1]:
        return similarity_matrix[func1][func2]
    elif func2 in similarity_matrix and func1 in similarity_matrix[func2]:
        return similarity_matrix[func2][func1]

    # Если нет в матрице - низкая схожесть
    return 0.0

# 5. Рассчитываем схожесть для каждой строки
results = []
for idx, row in data.iterrows():
    territory_func = row['Планируемое функциональное использование территории']

    # Если значение территории отсутствует — пропускаем (None)
    if pd.isna(territory_func):
        similarity = None
    # Если пользователь ничего не ввел
    elif not user_functions:
        similarity = None
    else:
        # Рассчитываем максимальную схожесть с введенными функциями
        max_similarity = None
        for user_func in user_functions:
            sim = calculate_function_similarity(territory_func, user_func)
            if sim is not None:
                if max_similarity is None or sim > max_similarity:
                    max_similarity = sim

        # Если все введённые функции некорректны → 0%
        similarity = max_similarity * 100 if max_similarity is not None else 0.0

        # Если нашлось хотя бы одно сравнение
        if max_similarity is not None:
            similarity = max_similarity * 100  # В процентах
        else:
            similarity = None

    results.append({
        'ID': row['ID'],
        'Сходство %': round(similarity, 3) if similarity is not None else None,
        'Планируемое функциональное использование территории': territory_func,
        'Триггер': row['Триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 6. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 МАКСИМАЛЬНО ПОХОЖИХ ТЕРРИТОРИЙ ПО ПЛАНИРУЕМОМУ НАЗНАЧЕНИЮ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 7. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Планируемое функциональное использование территории'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


Шаг 4: функциональное использование окружения

In [ ]:
# Код для сравнения по столбцу "Функциональное использование окружения" нечёткий коэффициент Жаккара

# 1. Выводим список функций
print("ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ТЕРРИТОРИИ:")
print_categories()
print("\n" + "="*80)

# 2. Пользователь вводит функции прилегающих территорий
print("\nВВЕДИТЕ ФУНКЦИОНАЛЬНОЕ ИСПОЛЬЗОВАНИЕ ПРИЛЕГАЮЩИХ ТЕРРИТОРИЙ В РАДИУСЕ 500 МЕТРОВ:")
print("(можно ввести несколько функций через запятую)")
user_input = input(">>> ").strip()

# Разделяем введенные функции
user_functions = [f.strip().lower() for f in user_input.split(',') if f.strip()]
print(f"\nПоиск по функциям: {user_functions}")
print("\n"+"="*80)

# 3. Проверяем, существует ли матрица сходства (из шага 2)
if 'similarity_matrix' not in globals():
    # Если по какой-то причине матрицы нет – выводим ошибку и завершаем
    raise ValueError("Матрица сходства не найдена. Сначала выполните шаг 2 (существующее использование).")

# 4. Функция для нечёткого Жаккара с использованием матрицы сходства
def fuzzy_jaccard(set1, set2, sim_matrix):
    """
    Рассчитывает нечёткий коэффициент Жаккара между двумя множествами функций.
    Используется матрица попарного сходства функций.
    """
    if set1 is None or set2 is None:
        return None
    if not set1 or not set2:
        return 0.0

    # Для каждого элемента из set1 находим максимальное сходство с элементами set2
    sum1 = 0.0
    for a in set1:
        max_sim = 0.0
        for b in set2:
            # Получаем сходство из матрицы
            if a in sim_matrix and b in sim_matrix[a]:
                sim = sim_matrix[a][b]
            elif b in sim_matrix and a in sim_matrix[b]:
                sim = sim_matrix[b][a]
            else:
                # Если функции нет в матрице – сходство 0 (неизвестная категория)
                sim = 0.0
            if sim > max_sim:
                max_sim = sim
        sum1 += max_sim

    # Для каждого элемента из set2 находим максимальное сходство с элементами set1
    sum2 = 0.0
    for b in set2:
        max_sim = 0.0
        for a in set1:
            if a in sim_matrix and b in sim_matrix[a]:
                sim = sim_matrix[a][b]
            elif b in sim_matrix and a in sim_matrix[b]:
                sim = sim_matrix[b][a]
            else:
                sim = 0.0
            if sim > max_sim:
                max_sim = sim
        sum2 += max_sim

    # Нечёткий Жаккар = (сумма max_sim для A + сумма max_sim для B) / (|A|+|B|)
    denominator = len(set1) + len(set2)
    if denominator == 0:
        return 0.0
    return (sum1 + sum2) / denominator

# 5. Предобработка строки функций из таблицы
def preprocess_functions(func_string):
    """Преобразует строку функций из таблицы в множество."""
    if pd.isna(func_string):
        return None
    # Разделяем по запятой, чистим пробелы, приводим к нижнему регистру
    functions = [f.strip().lower() for f in str(func_string).split(',')]
    # Удаляем пустые строки на случай повторов запятых
    functions = [f for f in functions if f]
    return set(functions)

# 6. Подготавливаем множество пользовательских функций
user_set = set(user_functions) if user_functions else None

# 7. Рассчитываем сходство для каждой строки
results = []
for idx, row in data.iterrows():
    # Получаем функции окружения из таблицы
    territory_funcs_raw = row['Функциональное использование окружения']
    territory_set = preprocess_functions(territory_funcs_raw)

    # Если пользователь не ввёл ничего или у территории нет данных
    if user_set is None or territory_set is None:
        similarity = None
    else:
        similarity = fuzzy_jaccard(user_set, territory_set, similarity_matrix)

    # Отображаем процент сходства
    similarity_percent = round(similarity * 100, 1) if similarity is not None else None

    # Для отображения функций территории в удобном виде
    territory_funcs_display = ', '.join(sorted(territory_set)) if territory_set else None

    results.append({
        'ID': row['ID'],
        'Сходство %': similarity_percent,
        'Функциональное использование окружения': territory_funcs_display,
        'Триггер': row['Триггер']
    })

# Создаём DataFrame с результатами
results_df = pd.DataFrame(results)

# 8. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ФУНКЦИЙ ОКРУЖЕНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 9. Сохраняем результаты для финального сравнения
if 'similarity_scores' not in globals():
    similarity_scores = {}

# Сохраняем коэффициенты сходства (уже в долях, делить не нужно)
similarity_scores['Функциональное использование окружения'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("Результаты сохранены. Можете перейти к следующему критерию.")


Шаг 5: медианное значение годов застройки окружения

In [ ]:
# Код для сравнения по столбцу "Год застройки окружения" логарифмическая нормализованная разница

# 1. Пользователь вводит средний год постройки окружения
print("\nВВЕДИТЕ МЕДИАННОЕ ЗНАЧЕНИЕ ГОДОВ ЗАСТРОЙКИ В РАДИУСЕ 500 МЕТРОВ")

# Проверяем, что введено целое число
user_year = None
while True:
    user_input = input(">>> ").strip()
    if user_input == "":
        break
    try:
        user_year = int(user_input)
        break
    except ValueError:
        print("Ошибка: введите целое число или оставьте поле пустым.")

print(f"\nСредний год застройки окружения: {user_year}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по годам
def calculate_year_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None
    # Проверяем, чтобы значения были положительными
    if user_val <= 0 or table_val <= 0 or min_val <= 0 or max_val <= 0:
        return None

    # Рассчитываем логарифмическую нормализованную разницу
    log_user = np.log(user_val)
    log_table = np.log(table_val)
    log_min = np.log(min_val)
    log_max = np.log(max_val)

    # Формула: 1 - (|log(PA) - log(PB)|) / (log(Pmax) - log(Pmin))
    similarity = 1 - (abs(log_user - log_table) / (log_max - log_min))
    # Обрезаем до [0,1] на случай выхода за пределы
    return max(0.0, min(1.0, similarity))

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение года в таблице
valid_years = data['Год застройки окружения'].dropna()
if len(valid_years) > 0:
    min_year = valid_years.min()
    max_year = valid_years.max()
else:
    min_year = max_year = None

for idx, row in data.iterrows():
    # Получаем год из таблицы
    table_year = row['Год застройки окружения']


    # Рассчитываем схожесть
    if user_year is None or pd.isna(table_year) or min_year is None:
        similarity = None
    else:
        similarity = calculate_year_similarity(user_year, table_year, min_year, max_year)

    # Преобразуем в проценты, если не None
    similarity_percent = round(similarity * 100, 1) if similarity is not None else pd.NA

    results.append({
        'ID': row['ID'],
        'Сходство %': similarity_percent,
        'Год застройки окружения': table_year,
        'Триггер': row['Триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ СРЕДНЕГО ГОДА ЗАСТРОЙКИ ОКРУЖЕНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 6. Сохраняем результаты для финального сравнения
# Создаем или обновляем словарь для хранения всех коэффициентов сходства
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Год застройки окружения'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


Шаг 6: коэффициент застройки окружения

In [ ]:
# Код для сравнения по столбцу "Коэффициент застройки" нормализованная разница

# 1. Пользователь вводит средний год постройки окружения
print("\nВВЕДИТЕ КОЭФФИЦИЕНТ ЗАСТРОЙКИ В РАДИУСЕ 500 МЕТРОВ")

# Проверяем, что введено целое число
user_coef = None
while True:
    user_input = input(">>> ").strip()
    if user_input == "":
        break
    # Заменяем запятую на точку
    user_input = user_input.replace(',', '.')
    try:
        val = float(user_input)
        if 0 <= val <= 1:
            user_coef = val
            break
        else:
            print("Ошибка: число должно быть от 0 до 1 или оставьте поле пустым.")
    except ValueError:
        print("Ошибка: введите число (например, 0.25) или оставьте поле пустым.")

print(f"\nКоэффициент застройки: {user_coef}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по коэффициентам
def calculate_coefficient_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None
    # Проверяем, чтобы значения находятся в допустимом диапазоне [0,1]
    if (1 <= user_val <= 0) or (1 <= table_val <= 0) or (1 <= min_val <= 0) or (1 <= max_val <= 0):
        return None

    # Рассчитываем нормализованную разницу
    # Формуле: 1 - (|PA - PB|) / (Pmax - Pmin)
    similarity = 1 - (abs(user_val - table_val) / (max_val - min_val))

    return max(0.0, min(1.0, similarity))

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение коэффициента в таблице
valid_coefs = data['Коэффициент застройки'].dropna()
if len(valid_coefs) > 0:
    min_coef = valid_coefs.min()
    max_coef = valid_coefs.max()
else:
    min_coef = max_coef = None

for idx, row in data.iterrows():
    table_coef = row['Коэффициент застройки']

    if user_coef is None or pd.isna(table_coef) or min_coef is None:
        similarity = None
    else:
        similarity = calculate_coefficient_similarity(user_coef, table_coef, min_coef, max_coef)

    # Преобразуем в проценты, если не None (используем NaN вместо pd.NA)
    similarity_percent = round(similarity * 100, 1) if similarity is not None else float('nan')

    results.append({
        'ID': row['ID'],
        'Сходство %': similarity_percent,
        'Коэффициент застройки': table_coef,
        'Триггер': row['Триггер']
    })

results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ КОЭФФИЦИЕНТА ЗАСТРОЙКИ ОКРУЖЕНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Коэффициент застройки'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


Шаг 7: коэффициент центральности (расположения территории в границах города)

In [ ]:
# Код для сравнения по столбцу "Коэффициент центральности" нормализованная разница

# 1. Пользователь вводит коэффициент центральности
print("\nВВЕДИТЕ ОТНОШЕНИЕ РАССТОЯНИЯ ОТ ЦЕНТРА ГОРОДА ДО ТЕРРИТОРИИ К РАССТОЯНИЮ ОТ ЦЕНТРА ГОРОДА ДО ГРАНИЦЫ ГОРОДА ПО ТОМУ ЖЕ НАПРАВЛЕНИЮ")

# Проверяем, что введено число (с заменой запятой на точку)
user_centr = None
while True:
    user_input = input(">>> ").strip()
    if user_input == "":
        break
    # Заменяем запятую на точку
    user_input = user_input.replace(',', '.')
    try:
        val = float(user_input)
        if val <= 1:
            user_centr = val
            break
        else:
            print("Ошибка: число должно быть меньше 1 или оставьте поле пустым.")
    except ValueError:
        print("Ошибка: введите число (например, 0.85) или оставьте поле пустым.")

print(f"\nКоэффициент центральности: {user_centr}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по коэффициентам (нормализованная разница)
def calculate_coefficient_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None
    # Проверяем, чтобы значения находились в допустимом диапазоне [0,1]
    if 1 <= user_val or 1 <= table_val or 1 <= min_val or 1 <= max_val:
        return None

    # Рассчитываем нормализованную разницу
    # Формула: 1 - (|PA - PB|) / (Pmax - Pmin)
    similarity = 1 - (abs(user_val - table_val) / (max_val - min_val))

    return max(0.0, min(1.0, similarity))

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение коэффициента в таблице
valid_centr = data['Коэффициент центральности'].dropna()
if len(valid_centr) > 0:
    min_centr = valid_centr.min()
    max_centr = valid_centr.max()
else:
    min_centr = max_centr = None

for idx, row in data.iterrows():
    table_centr = row['Коэффициент центральности']

    if user_centr is None or pd.isna(table_centr) or min_centr is None:
        similarity = None
    else:
        similarity = calculate_coefficient_similarity(user_centr, table_centr, min_centr, max_centr)

    # Преобразуем в проценты, если не None (используем NaN)
    similarity_percent = round(similarity * 100, 1) if similarity is not None else float('nan')

    results.append({
        'ID': row['ID'],
        'Сходство %': similarity_percent,
        'Коэффициент центральности': table_centr,
        'Триггер': row['Триггер']
    })

results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ КОЭФФИЦИЕНТА ЦЕНТРАЛЬНОСТИ ОКРУЖЕНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Коэффициент центральности'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")


Шаг 8: численность населения города

In [ ]:
# Код для сравнения по столбцу "Население" логарифмическая нормализованная разница

# 1. Пользователь вводит численность населения
print("\nВВЕДИТЕ ЧИСЛЕННОСТЬ НАСЕЛЕНИЯ ГОРОДА")

# Проверяем, что введено целое число
user_pop = None
while True:
    user_input = input(">>> ").strip()
    if user_input == "":
        break
    try:
        user_pop = int(user_input)
        break
    except ValueError:
        print("Ошибка: введите целое число или оставьте поле пустым.")

print(f"\nЧисленность населения: {user_pop}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по численности населения
def calculate_population_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None
    # Проверяем, чтобы значения были положительными
    if user_val <= 0 or table_val <= 0 or min_val <= 0 or max_val <= 0:
        return None

    # Рассчитываем логарифмическую нормализованную разницу
    log_user = np.log(user_val)
    log_table = np.log(table_val)
    log_min = np.log(min_val)
    log_max = np.log(max_val)

    # Формула: 1 - (|log(PA) - log(PB)|) / (log(Pmax) - log(Pmin))
    similarity = 1 - (abs(log_user - log_table) / (log_max - log_min))
    # Обрезаем до [0,1] на случай выхода за пределы
    return max(0.0, min(1.0, similarity))

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение численности населения в таблице
valid_pop = data['Население'].dropna()
if len(valid_pop) > 0:
    min_pop = valid_pop.min()
    max_pop = valid_pop.max()
else:
    min_pop = max_pop = None

for idx, row in data.iterrows():
    # Получаем численность населения из таблицы
    table_pop = row['Население']

    # Рассчитываем схожесть
    if user_pop is None or pd.isna(table_pop) or min_pop is None:
        similarity = None
    else:
        similarity = calculate_population_similarity(user_pop, table_pop, min_pop, max_pop)

    # Преобразуем в проценты, если не None
    similarity_percent = round(similarity * 100, 1) if similarity is not None else pd.NA

    results.append({
        'ID': row['ID'],
        'Сходство %': similarity_percent,
        'Население': table_pop,
        'Триггер': row['Триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ПО ЧИСЛЕННОСТИ НАСЕЛЕНИЯ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Население'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")

Шаг 9: изменение существующей застройки

In [ ]:
# Код для сравнения по столбцу "Год постройки ОКС" логарифмическая нормализованная разница

# 1. Пользователь вводит год строительства здания
print("\nВВЕДИТЕ ГОД СТРОИТЕЛЬСТВА СУЩЕСТВУЮЩЕГО И ИЗМЕНЯЕМОГО ПРОЕКТОМ ОКС НА ТЕРРИТОРИИ")
print("(если застройка не изменяется, оставьте поле пустым)")

# Проверяем, что введено целое число
user_year = None
while True:
    user_input = input(">>> ").strip()
    if user_input == "":
        break
    try:
        user_year = int(user_input)
        break
    except ValueError:
        print("Ошибка: введите целое число или оставьте поле пустым.")

print(f"\nГод постройки ОКС: {user_year}")
print("\n" + "="*80)

# 2. Функция для расчета сходства по годам
def calculate_year_similarity(user_val, table_val, min_val, max_val):

    # Если хотя бы одно значение отсутствует
    if pd.isna(user_val) or pd.isna(table_val):
        return None
    # Проверяем, чтобы значения были положительными
    if user_val <= 0 or table_val <= 0 or min_val <= 0 or max_val <= 0:
        return None

    # Рассчитываем логарифмическую нормализованную разницу
    log_user = np.log(user_val)
    log_table = np.log(table_val)
    log_min = np.log(min_val)
    log_max = np.log(max_val)

    # Формула: 1 - (|log(PA) - log(PB)|) / (log(Pmax) - log(Pmin))
    similarity = 1 - (abs(log_user - log_table) / (log_max - log_min))
    # Обрезаем до [0,1] на случай выхода за пределы
    return max(0.0, min(1.0, similarity))

# 3. Рассчитываем схожесть для каждой строки
results = []

# Определяем минимальное и максимальное значение года в таблице
valid_years = data['Год постройки ОКС'].dropna()
if len(valid_years) > 0:
    min_year = valid_years.min()
    max_year = valid_years.max()
else:
    min_year = max_year = None

for idx, row in data.iterrows():
    # Получаем год из таблицы
    table_year = row['Год постройки ОКС']

    # Рассчитываем схожесть
    if user_year is None or pd.isna(table_year) or min_year is None:
        similarity = None
    else:
        similarity = calculate_year_similarity(user_year, table_year, min_year, max_year)

    # Преобразуем в проценты, если не None
    similarity_percent = round(similarity * 100, 1) if similarity is not None else pd.NA

    results.append({
        'ID': row['ID'],
        'Сходство %': similarity_percent,
        'Год постройки ОКС': table_year,
        'Триггер': row['Триггер']
    })

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)

# 4. Сортируем по убыванию сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ГОДА ПОСТРОЙКИ ОКС:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 5. Сохраняем результаты для финального сравнения
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Год постройки ОКС'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("\nРезультаты сохранены. Можете перейти к следующему критерию.")

Шаг 10: дополнительная информация

In [ ]:
# Код для сравнения по бинарным признакам индекс Рэнда

# 1. Список бинарных столбцов (порядок важен, но не критичен)
binary_columns = [
    'Новое строительство',
    'Снос',
    'Исторические здания',
    'Железная дорога',
    'Автодорога',
    'Инженерные объекты',
    'Водные объекты'
]

print("\nЗАПОЛНИТЕ ДОПОЛНИТЕЛЬНУЮ ИНФОРМАЦИЮ")
print("Для каждого признака введите 1-да, 0-нет или оставьте пустым (исключение из расчета).")
print("="*80)

# 2. Сбор ответов пользователя с проверкой ввода
user_binary = {}
for col in binary_columns:
    # Формируем понятный вопрос
    if col == 'Новое строительство':
        prompt = "Строительство на ранее свободной от застройки территории?"
    elif col == 'Снос':
        prompt = "Снос существующих ОКС на территории?"
    elif col == 'Исторические здания':
        prompt = "Исторические здания в окружении радиусом 500 метров?"
    elif col == 'Железная дорога':
        prompt = "Железная дорога в окружении радиусом 500 метров?"
    elif col == 'Автодорога':
        prompt = "Магистральная автодорога в окружении радиусом 500 метров?"
    elif col == 'Инженерные объекты':
        prompt = "Инженерные объекты в окружении радиусом 500 метров?"
    elif col == 'Водные объекты':
        prompt = "Водные объекты в окружении радиусом 500 метров?"
    else:
        prompt = f"Наличие {col}?"

    print(f"\n{prompt}")

    value = None
    while True:
        user_input = input(">>> ").strip()
        if user_input == "":
            break
        if user_input in ('1', '0'):
            value = int(user_input)
            break
        else:
            print("Ошибка: введите 1, 0 или оставьте поле пустым.")

    user_binary[col] = value
    # Выводим принятое значение
    if value is not None:
        print(f"{col}: {value}")
    else:
        print(f"{col}: ")

print("\n" + "="*80)

# 3. Функция для расчёта индекса Рэнда с заменой NaN -> 0
def rand_index_with_nan_as_zero(user_vec, territory_vec):
    """
    Вычисляет индекс Рэнда для бинарных векторов.
    NaN в территории трактуется как 0 (отсутствие признака).
    Пропуски у пользователя (None) исключаются из сравнения.
    """
    a = b = c = d = 0
    n_compared = 0

    for col in binary_columns:
        u = user_vec.get(col)
        if u is None:
            continue  # пользователь не знает, не учитываем

        t = territory_vec.get(col)
        if pd.isna(t):
            t = 0  # в таблице NaN → признак отсутствует

        n_compared += 1
        if u == 1 and t == 1:
            a += 1
        elif u == 1 and t == 0:
            b += 1
        elif u == 0 and t == 1:
            c += 1
        else:  # u == 0 and t == 0
            d += 1

    if n_compared == 0:
        return None
    return (a + d) / (a + b + c + d)

# 4. Расчёт для всех территорий
print("\nРАСЧЁТ СХОДСТВА ПО БИНАРНЫМ ПРИЗНАКАМ")
print("-" * 80)

results = []
for idx, row in data.iterrows():
    territory_vec = {col: row[col] for col in binary_columns}
    rand = rand_index_with_nan_as_zero(user_binary, territory_vec)

    similarity_percent = round(rand * 100, 1) if rand is not None else None
    results.append({
        'ID': row['ID'],
        'Сходство %': similarity_percent,
        'Триггер': row['Триггер']
    })

results_df = pd.DataFrame(results)

# 5. Топ-10
print("\nТОП-10 ТЕРРИТОРИЙ С НАИБОЛЬШИМ СХОДСТВОМ ПО БИНАРНЫМ ПРИЗНАКАМ:")
print("-" * 80)
top_10 = results_df.sort_values('Сходство %', ascending=False).head(10)
print(top_10.to_string(index=False))

# 6. Сохраняем в словарь (в долях, не процентах)
if 'similarity_scores' not in globals():
    similarity_scores = {}

similarity_scores['Бинарные признаки (Рэнда)'] = dict(zip(results_df['ID'], results_df['Сходство %'] / 100))

print("\n" + "="*80)
print("Результаты по бинарным признакам 0сохранены. Можете перейти к итоговому сходству.")

Шаг 11: пространственно-функциональное подобные территории

In [ ]:
# итоговое подобие территорий композитное взвешенное сходство

criteria_list = [
    'Существующее функциональное использование территории',
    'Планируемое функциональное использование территории',
    'Функциональное использование окружения',
    'Год застройки окружения',
    'Коэффициент застройки',
    'Коэффициент центральности',
    'Население',
    'Год постройки ОКС',
    'Бинарные признаки'
]

# 1. Веса по умолчанию (предложенные экспертом)
default_weights = {
    'Существующее функциональное использование территории': 0.20,
    'Планируемое функциональное использование территории': 0.20,
    'Функциональное использование окружения': 0.15,
    'Год застройки окружения': 0.10,
    'Коэффициент застройки': 0.10,
    'Коэффициент центральности': 0.15,
    'Население': 0.04,
    'Год постройки ОКС': 0.04,
    'Бинарные признаки': 0.02
}

# Проверяем, что сумма весов равна 1
total_weight = sum(default_weights.values())
if abs(total_weight - 1.0) > 0.001:
    # Нормализуем веса, если сумма не равна 1
    default_weights = {k: v/total_weight for k, v in default_weights.items()}

print("ВЕСА КРИТЕРИЕВ ПО УМОЛЧАНИЮ (экспертные оценки):")
print("-" * 80)
for criterion in criteria_list:
    weight = default_weights[criterion]
    print(f"{criterion}: {weight:.3f}")

print(f"\nСумма весов: {sum(default_weights.values()):.3f}")
print("\n" + "="*80)

# 2. Запрос на изменение весов с проверкой ввода
print("\nХотите изменить веса критериев?")
print("Введите 1-да или 0-нет")
change_input = input(">>> ").strip()

change_weights = None
if change_input:
    if change_input == '1':
        change_weights = 1
        print(f"\nИзменение весов: включено")
    elif change_input == '0':
        change_weights = 0
        print(f"\nИзменение весов: отключено")
    else:
        print("Ошибка: введите '1' или '0'. Используются веса по умолчанию.")
        change_weights = 0
else:
    print("Используются веса по умолчанию.")
    change_weights = 0

print("\n" + "="*80)

# 3. Настройка пользовательских весов
if change_weights:
    print("\nВВЕДИТЕ ВЕСА ДЛЯ КАЖДОГО КРИТЕРИЯ (от 0 до 1)")
    print("Сумма всех весов должна быть равна 1")
    print("-" * 80)

    custom_weights = {}
    for criterion in criteria_list:

        while True:
            weight_input = input(f"Вес для '{criterion}': ").strip()
            # Проверка на пустой ввод
            if not weight_input:
                print("  Ошибка: введите число от 0 до 1")
                continue
            try:
                weight = float(weight_input)
                # Проверка диапазона
                if weight < 0 or weight > 1:
                    print("  Ошибка: введите число от 0 до 1")
                    continue

                custom_weights[criterion] = weight
                print(f"  ✓ Установлен вес: {weight:.3f}")
                break

            except ValueError:
                print("  Ошибка: введите число от 0 до 1")

    print("\n" + "-" * 80)
    print("ВВЕДЕННЫЕ ВЕСА:")
    print("-" * 80)

    # Суммируем введенные веса
    total_weight = sum(custom_weights.values())
    print(f"Сумма введенных весов: {total_weight:.3f}")

    if abs(total_weight - 1.0) > 0.001:
        # Нормализуем веса
        custom_weights = {k: v/total_weight for k, v in custom_weights.items()}

    print("\nНОРМАЛИЗОВАННЫЕ ВЕСА:")
    print("-" * 80)
    for criterion in criteria_list:
        weight = custom_weights[criterion]
        print(f"{criterion}: {weight:.3f}")

    print(f"\nСумма весов после нормализации: {sum(custom_weights.values()):.3f}")

    weights = custom_weights

else:
    print("\nИспользуются веса по умолчанию")
    weights = default_weights

print("\n" + "="*80)
print("\nРАСЧЕТ СХОДСТВА")
print("-" * 80)

# 4. Функция для расчета итогового сходства
def calculate_total_similarity(row_id, weights_dict):
    """Рассчитывает итоговое взвешенное сходство для конкретного ID"""
    total_similarity = 0
    total_weight = 0
    details = {}

    for criterion, weight in weights_dict.items():
        if criterion in similarity_scores and row_id in similarity_scores[criterion]:
            similarity = similarity_scores[criterion][row_id]
            if similarity is not None:
                try:
                    # Преобразуем в число и проверяем на NaN
                    similarity_float = float(similarity)
                    if not np.isnan(similarity_float):
                        total_similarity += similarity_float * weight
                        total_weight += weight
                        details[criterion] = similarity_float
                    else:
                        details[criterion] = None
                except (ValueError, TypeError):
                    details[criterion] = None
            else:
                details[criterion] = None
        else:
            details[criterion] = None

    # Если нет ни одного критерия с данными
    if total_weight == 0:
        return None, details

    final_similarity = total_similarity / total_weight
    return final_similarity, details

# 5. Рассчитываем итоговое сходство для всех строк
final_results = []

for idx, row in data.iterrows():
    row_id = row['ID']
    final_similarity, details = calculate_total_similarity(row_id, weights)

    # Подсчитываем количество критериев с данными
    criteria_with_data = sum(1 for val in details.values() if val is not None)

    final_results.append({
        'ID': row_id,
        'Триггер': row['Триггер'],
        'Итоговое сходство %': round(final_similarity * 100, 1) if final_similarity is not None else None,
        'Критериев с данными': criteria_with_data,
        **{f'{criterion} %': round(details[criterion] * 100, 1) if details[criterion] is not None else None
           for criterion in criteria_list}
    })

# Создаем DataFrame с результатами
final_df = pd.DataFrame(final_results)

# 6. Сортируем по убыванию итогового сходства и показываем топ-10
print("\nТОП-10 ТЕРРИТОРИЙ ПО ПРОСТРАНСТВЕННОМУ СХОДСТВУ:")
print("(сортировка по убыванию итогового сходства)")
print("-" * 100)

# Создаем список отображаемых колонок, сортируем и форматируем
display_columns = ['ID', 'Триггер', 'Итоговое сходство %', 'Критериев с данными'] + [f'{criterion} %' for criterion in criteria_list]
top_10_final = final_df.sort_values('Итоговое сходство %', ascending=False).head(10)

def format_cell(value, col_name):
    if col_name.endswith(' %') and value is not None and isinstance(value, (int, float)):
        return f"{value}%"
    elif value is None:
        return "-"
    else:
        return str(value)

# Форматируем каждую ячейку отдельно
formatted_rows = []
for _, row in top_10_final[display_columns].iterrows():
    formatted_row = []
    for col in display_columns:
        value = row[col]
        formatted_row.append(format_cell(value, col))
    formatted_rows.append(formatted_row)

# Выводим таблицу с выравниванием
from tabulate import tabulate
print(tabulate(formatted_rows, headers=display_columns, tablefmt='grid'))

# 7. Выводим детализацию для лучшего результата
print("\n" + "="*80)
print("ДЕТАЛИЗАЦИЯ ЛУЧШЕГО РЕЗУЛЬТАТА:")
print("-" * 80)

best_result = top_10_final.iloc[0]
print(f"ID: {best_result['ID']}")
print(f"Триггер конфликта: {best_result['Триггер']}")
print(f"Итоговое сходство: {best_result['Итоговое сходство %']}%")
print(f"Учтено критериев: {best_result['Критериев с данными']}/{len(criteria_list)}")
print("\nСходство по критериям:")
for criterion in criteria_list:
    similarity = best_result[f'{criterion} %']
    weight = weights[criterion]
    if similarity is not None:
        print(f"  {criterion}: {similarity}% (вес: {weight:.3f})")

print("\n" + "="*80)
print("РАСЧЕТ ЗАВЕРШЕН. НАЙДЕНО 10 НАИБОЛЕЕ ПОХОЖИХ ТЕРРИТОРИЙ.")

# 8. Сохраняем словарь с весами для возможного использования
if 'criteria_weights' not in globals():
    criteria_weights = {}
criteria_weights = weights.copy()


Шаг 12: сопоставление конфликтологической экспертизы с пространственно-функциональными характеристиками территории

In [ ]:
# Загрузка и обработка файла с конфликтами

print("ЗАГРУЗКА ДАННЫХ О КОНФЛИКТАХ")
print("-" * 80)

# 1. Проверяем наличие файла, если нет – загружаем
if not os.path.exists('conflict.xlsx'):
    print("Файл conflict.xlsx не найден.")
    print("Пожалуйста, загрузите файл с конфликтами (Excel с именем conflict.xlsx).")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn != 'conflict.xlsx':
            os.rename(fn, 'conflict.xlsx')
            print(f"Файл {fn} переименован в conflict.xlsx")
else:
    print("Файл conflict.xlsx найден.")

# Загружаем данные
df_con = pd.read_excel('conflict.xlsx')
print(f"Количество строк: {df_con.shape[0]}, столбцов: {df_con.shape[1]}")

# 2. Проверяем наличие столбца ID
if 'ID' not in df_con.columns:
    print("\nВНИМАНИЕ: столбец ID не найден.")
    id_col = input("Введите название столбца, содержащего ID: ").strip()
    if id_col in df_con.columns:
        df_con = df_con.rename(columns={id_col: 'ID'})
        print(f"Столбец '{id_col}' переименован в 'ID'")
    else:
        raise ValueError(f"Столбец '{id_col}' не найден в файле")

# Таблица с пропусками
print("\nПОЛУЧЕННЫЕ ДАННЫЕ:")
print("-" * 80)
info_df = pd.DataFrame({
    'Не пропущено': df_con.count(),
    'Пропущено': df_con.isnull().sum(),
    '% пропусков': (df_con.isnull().sum() / len(df_con) * 100).round(1)
})
print(info_df)

# Очистка текстовых полей
def clean_conflict_text(df):
    """Очищает текстовые столбцы: удаляет лишние пробелы, заменяет множественные пробелы на один."""
    df_clean = df.copy()
    text_cols = [col for col in df_clean.columns if df_clean[col].dtype == 'object' and col != 'ID']
    for col in text_cols:
        df_clean[col] = df_clean[col].astype(str).str.strip()
        df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)
        df_clean[col] = df_clean[col].replace(['nan', 'None', ''], pd.NA)
    return df_clean

df_con = clean_conflict_text(df_con)

print("\n" + "-" * 80)
print("Данные о конфликтах загружены и очищены.")

# 3. Вывод информации о 3 самых похожих конфликтах
print("\n" + "=" * 80)
print("ПОДРОБНАЯ ИНФОРМАЦИЯ ПО 3 САМЫМ ПОХОЖИМ КОНФЛИКТАМ")
print("-" * 80)

for i, (_, row) in enumerate(top_10_final.head(3).iterrows(), 1):
    conflict_id = row['ID']
    similarity = row['Итоговое сходство %']   # обратите внимание: столбец называется "Итоговое сходство %" (с пробелом)
    trigger = row['Триггер']

    print(f"\n{'='*80}")
    print(f"Конфликт #{i} | ID: {conflict_id}")
    print(f"Сходство с проектом: {similarity}%")
    print(f"{'-'*80}")

    # Ищем запись в df_con
    conflict_details = df_con[df_con['ID'] == conflict_id]
    if not conflict_details.empty:
        conflict_row = conflict_details.iloc[0]
        # Выводим все непустые поля, кроме ID и слишком длинных
        for col in df_con.columns:
            if col == 'ID':
                continue
            val = conflict_row[col]
            if pd.isna(val):
                continue
            # Преобразуем числа с плавающей точкой в целые, если возможно
            if isinstance(val, float) and val.is_integer():
                val = int(val)
            # Длинные тексты разбиваем на строки
            if isinstance(val, str) and len(val) > 100:
                print(f"\n{col}:")
                words = val.split()
                line = ""
                for word in words:
                    if len(line) + len(word) + 1 > 100:
                        print(line)
                        line = word
                    else:
                        line += " " + word if line else word
                if line:
                    print(line)
            else:
                print(f"\n{col}: {val}")
    else:
        print(f"\nДополнительная информация для конфликта ID {conflict_id} не найдена.")

    # Выводим сходство по критериям
    print(f"\nСходство по критериям:")
    for criterion in criteria_list:
        # В top_10_final столбцы-критерии имеют имена вида 'criterion %'
        col_name = f'{criterion} %'
        if col_name in row.index:
            sim_val = row[col_name]
            if sim_val is not None:
                print(f"  - {criterion}: {sim_val}% (вес: {weights.get(criterion, 0):.3f})")

print("\n" + "=" * 80)

# 4. Сопоставление топ-10 с дополнительной информацией
print("\nДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ ПО ТОП-10 ТЕРРИТОРИЙ (сопоставление с конфликтами)")
print("-" * 80)

# Список желаемых столбцов из конфликтного файла
desired_conflict_cols = [
    'Описание конфликта',
    'Триггеры',
    'Акторы',
    'Организаторы',
    'Интересанты проекта',
    'Форма протеста',
    'Исход конфликта',
    'Масштаб конфликта',
    'Город',
    'Адрес',
    'Год начала конфликта',
    'Год окончания конфликта'
]

# Оставляем только те, что есть в df_con
available_conflict_cols = [col for col in desired_conflict_cols if col in df_con.columns]

if available_conflict_cols:
    # Объединяем top_10_final с конфликтными данными
    merge_cols = ['ID'] + available_conflict_cols
    top_10_with_conflict = pd.merge(
        top_10_final[['ID', 'Триггер', 'Итоговое сходство %']],
        df_con[merge_cols],
        on='ID',
        how='left'
    )

    # Переименовываем столбец сходства для наглядности
    top_10_with_conflict = top_10_with_conflict.rename(columns={'Итоговое сходство %': 'Сходство (%)'})

    # Обрезаем длинные тексты до 50 символов для компактного вывода
    def shorten_text(val, max_len=50):
        if isinstance(val, str) and len(val) > max_len:
            return val[:max_len] + '…'
        return val

    for col in available_conflict_cols:
        top_10_with_conflict[col] = top_10_with_conflict[col].apply(shorten_text)

    # Определяем порядок столбцов для вывода
    display_cols = ['ID', 'Сходство (%)', 'Триггер'] + available_conflict_cols
    # Оставляем только существующие
    display_cols = [col for col in display_cols if col in top_10_with_conflict.columns]

    print("\nТАБЛИЦА ТОП-10 ТЕРРИТОРИЙ С ИНФОРМАЦИЕЙ О КОНФЛИКТАХ:")
    print("-" * 100)
    print(top_10_with_conflict[display_cols].to_string(index=False))

else:
    print("Не найдено ни одного из стандартных столбцов в файле conflict.xlsx.")
    print("Доступные столбцы:", list(df_con.columns))

print("\n" + "=" * 80)
print("АНАЛИЗ ЗАВЕРШЕН.")


Шаг 13: скачивание полной сводной таблицы лучших совпадений

In [ ]:
# СОЗДАНИЕ ФИНАЛЬНОГО CSV ФАЙЛА

print("СОЗДАНИЕ ФИНАЛЬНОГО CSV ФАЙЛА")
print("=" * 80)

# 1. Режим сохранения
print("\nВЫБЕРИТЕ РЕЖИМ СОХРАНЕНИЯ:")
print("0 - Сохранить только топ-10 лучших совпадений")
print("1 - Сохранить ВСЕ рассчитанные конфликты")
save_mode = input("Введите 0 или 1: ").strip()

# Определяем источник данных
if save_mode == '1':
    source_df = final_df.sort_values('Итоговое сходство %', ascending=False).copy()
    mode_name = "all"
    print("\nВыбран режим: СОХРАНИТЬ ВСЕ ДАННЫЕ")
else:
    source_df = top_10_final.copy()
    mode_name = "top10"
    print("\nВыбран режим: СОХРАНИТЬ ТОП-10")

# 2. Список столбцов из файла конфликтов (все, кроме ID)
conflict_cols = [col for col in df_con.columns if col != 'ID']

# 3. Сбор данных для выбранных ID
selected_ids = source_df['ID'].tolist()

# Берём строки из territory (data) и из конфликтов
selected_territory = data[data['ID'].isin(selected_ids)].copy()
selected_conflict = df_con[df_con['ID'].isin(selected_ids)].copy()

# Добавляем итоговое сходство к territory
selected_territory = pd.merge(
    selected_territory,
    source_df[['ID', 'Триггер', 'Итоговое сходство %']],
    on='ID',
    how='left'
)

# 4. Формирование финального DataFrame построчно
final_rows = []

for _, row in source_df.iterrows():
    conflict_id = row['ID']
    row_data = {}

    # ID, триггер, итоговое сходство
    row_data['ID'] = conflict_id
    row_data['Триггер'] = row['Триггер']
    row_data['Итоговое сходство %'] = row['Итоговое сходство %']

    # Все столбцы из conflict.xlsx (кроме ID)
    conflict_info = selected_conflict[selected_conflict['ID'] == conflict_id]
    if not conflict_info.empty:
        conf_row = conflict_info.iloc[0]
        for col in conflict_cols:
            row_data[col] = conf_row[col]

    # Данные по критериям (значение и сходство)
    territory_info = selected_territory[selected_territory['ID'] == conflict_id]
    if not territory_info.empty:
        terr_row = territory_info.iloc[0]
        for criterion in criteria_list:
            # Исходное значение критерия (как есть)
            row_data[f'{criterion} (значение)'] = terr_row.get(criterion, None)
            # Сходство по критерию в процентах
            sim_col = f'{criterion} %'
            row_data[f'{criterion} (сходство %)'] = row.get(sim_col, None)

    final_rows.append(row_data)

# Создаём DataFrame
final_df_export = pd.DataFrame(final_rows)

# 5. Определяем порядок столбцов (для удобства)
ordered_columns = ['ID', 'Триггер', 'Итоговое сходство %'] + conflict_cols
for criterion in criteria_list:
    ordered_columns.append(f'{criterion} (значение)')
    ordered_columns.append(f'{criterion} (сходство %)')
# Оставляем только существующие столбцы
ordered_columns = [col for col in ordered_columns if col in final_df_export.columns]
final_df_export = final_df_export[ordered_columns]

# 6. Сохраняем в CSV
from datetime import datetime
import os

if not source_df.empty:
    first_id = source_df.iloc[0]['ID']
    output_filename = f"{mode_name}_conflicts_{first_id}.csv"
    final_df_export.to_csv(output_filename, index=False, encoding='utf-8-sig')

    print(f"\nФайл успешно создан: {output_filename}")
    print(f"Количество строк: {len(final_df_export)}")
    print(f"Количество столбцов: {len(final_df_export.columns)}")

    print("\nСтруктура файла:")
    print("-" * 80)
    print("• ID конфликта")
    print("• Триггер")
    print("• Итоговое сходство %")
    print("• Данные о конфликте")
    print("• Данные о территории: значение + сходство (%)")

    print("\n" + "=" * 80)

# 7. Предпросмотр
    print("\nПРЕДПРОСМОТР (первые 2 строки):")
    print("-" * 80)

    preview_cols = final_df_export.columns
    print(final_df_export[preview_cols].head(2).to_string(index=False))
    print("\n" + "=" * 80)
else:
    print("Нет данных для сохранения.")
